In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb

df = pd.read_csv('../data/ald_process_data.csv')

# doi='DUMMY_DATA' 표시 (실데이터와 구분)
n_dummy = (df['doi'] == 'DUMMY_DATA').sum()
print(f"전체 데이터: {len(df)}행 (더미: {n_dummy}행)")

# 범주형 인코딩
cat_cols = ['material', 'precursor', 'oxidant']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[f'{col}_enc'] = le.fit_transform(df[col])
    encoders[col] = le
    print(f"{col} 인코딩: {dict(zip(le.classes_, le.transform(le.classes_)))}")

feature_cols = ['material_enc', 'precursor_enc', 'oxidant_enc',
                'temperature', 'pulse_time', 'purge_time']
X = df[feature_cols]
y = df['gpc']
print(f"\nfeature: {feature_cols}")
print(f"target 통계: mean={y.mean():.3f}, std={y.std():.3f}, min={y.min():.3f}, max={y.max():.3f}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)}  Test: {len(X_test)}")

model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=5,   # regularize for small dataset
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
model.fit(X_train, y_train,
          eval_set=[(X_test, y_test)],
          verbose=False)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)
print(f"\nMAE : {mae:.4f} Å/cycle")
print(f"R²  : {r2:.3f}")
print("(더미 데이터 기준 — 실데이터 교체 후 성능 재평가 필요)")
print("(목표: R² > 0.5, 실데이터로 달성 기대)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter: actual vs predicted
axes[0].scatter(y_test, y_pred, alpha=0.6, s=30, color='steelblue')
lims = [min(y_test.min(), y_pred.min()) - 0.05,
        max(y_test.max(), y_pred.max()) + 0.05]
axes[0].plot(lims, lims, 'r--', linewidth=1)
axes[0].set_xlabel('Actual GPC (Å/cycle)')
axes[0].set_ylabel('Predicted GPC (Å/cycle)')
axes[0].set_title(f'Process Model  MAE={mae:.3f}  R²={r2:.3f}\n(더미 데이터)')

# Feature importance
importance = pd.Series(model.feature_importances_, index=feature_cols)
importance.sort_values()[::-1].plot(kind='barh', ax=axes[1], color='seagreen')
axes[1].set_xlabel('Feature Importance (gain)')
axes[1].set_title('Top Features — GPC Model')

plt.tight_layout()
plt.savefig('../data/process_model_eval.png', dpi=150)
plt.show()

model.save_model('../data/xgb_process_model.json')
print("모델 저장 완료: data/xgb_process_model.json")